<a href="https://colab.research.google.com/github/financieras/antigen_predictor/blob/main/notebooks/04_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: App de Streamlit

**Entrada:** `model.pkl` (modelo entrenado en NB03)  
**Salida:** App web interactiva accesible desde el navegador

Este notebook lanza la aplicación Streamlit que permite predecir la antigenicidad
de proteínas a partir de secuencias en formato FASTA.

## Estructura de la app
- **Sidebar:** información del modelo (AUC-ROC, proteínas de entrenamiento, limitaciones)
- **Panel principal:** entrada de secuencias FASTA (texto o archivo)
- **Resultados:** gráfico de barras + tabla ordenada por score + descarga CSV

## Archivos necesarios
```
app/
├── app.py        ← código de la aplicación
└── model.pkl     ← modelo guardado en NB03
```

## 0. Configuración y montaje de Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Instalación de dependencias

In [2]:
!pip install streamlit biopython pyngrok --quiet
print("Dependencias instaladas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.7 MB/s eta 0:00:00
Dependencias instaladas.


## 2. Copiar model.pkl desde Drive al entorno local de Colab

In [3]:
import shutil
import os

# Crear carpeta de trabajo
os.makedirs('/content/app', exist_ok=True)

# Copiar modelo desde Drive
shutil.copy(
    '/content/drive/MyDrive/epitope/model.pkl',
    '/content/app/model.pkl'
)

print("model.pkl copiado a /content/app/")
print(f"Tamaño: {os.path.getsize('/content/app/model.pkl') / 1024:.1f} KB")

model.pkl copiado a /content/app/
Tamaño: 5274.5 KB


## 3. Escribir app.py

El código de la app se escribe directamente desde esta celda.
Es el mismo archivo `app/app.py` del repositorio.

In [4]:
%%writefile /content/app/app.py
"""
app.py — Clasificador de Antigenicidad de Proteínas
Proyecto educativo de Machine Learning aplicado a bioinformática.
"""

import streamlit as st
import joblib
import numpy as np
import re
import matplotlib.pyplot as plt
from io import StringIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# ── Configuración de página ──────────────────────────────────────────────────
st.set_page_config(
    page_title="Clasificador de Antigenicidad",
    page_icon="🧬",
    layout="wide"
)

# ── Cargar modelo ────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    bundle = joblib.load("model.pkl")
    return bundle

bundle = load_model()
model        = bundle["model"]
feature_cols = bundle["feature_cols"]
amino_acids  = bundle["amino_acids"]
label_map    = bundle["label_map"]
auc_cv       = bundle["auc_cv"]
n_train      = bundle["n_train"]

# ── Funciones ────────────────────────────────────────────────────────────────
def clean_sequence(seq: str) -> str:
    return re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq.upper().replace(" ", "").replace("\n", ""))


def compute_features(seq: str):
    seq = clean_sequence(seq)
    if len(seq) < 10:
        return None
    try:
        analysis = ProteinAnalysis(seq)
        features = {
            "length":            len(seq),
            "molecular_weight":  analysis.molecular_weight(),
            "isoelectric_point": analysis.isoelectric_point(),
            "gravy":             analysis.gravy(),
        }
        aa_comp = analysis.amino_acids_percent
        for aa in amino_acids:
            features[f"aa_{aa}"] = aa_comp.get(aa, 0.0)
        return features
    except Exception:
        return None


def parse_fasta(text: str):
    proteins = []
    current_name = None
    current_seq  = []
    for line in text.strip().splitlines():
        line = line.strip()
        if line.startswith(">"):
            if current_name is not None:
                proteins.append((current_name, "".join(current_seq)))
            current_name = line[1:].split()[0]
            current_seq  = []
        elif line:
            current_seq.append(line)
    if current_name is not None:
        proteins.append((current_name, "".join(current_seq)))
    return proteins


def predict_proteins(proteins):
    results = []
    for name, seq in proteins:
        feats = compute_features(seq)
        if feats is None:
            results.append({
                "Proteína": name,
                "Longitud": len(clean_sequence(seq)),
                "Score antigenicidad": None,
                "Predicción": "⚠️ Secuencia inválida o muy corta",
                "MW (Da)": None, "pI": None, "GRAVY": None,
            })
            continue
        X = np.array([[feats[col] for col in feature_cols]])
        score = model.predict_proba(X)[0][1]
        label = "🟢 Antigénica" if score >= 0.5 else "🔴 No antigénica"
        results.append({
            "Proteína":            name,
            "Longitud":            feats["length"],
            "Score antigenicidad": round(score, 4),
            "Predicción":          label,
            "MW (Da)":             round(feats["molecular_weight"], 1),
            "pI":                  round(feats["isoelectric_point"], 2),
            "GRAVY":               round(feats["gravy"], 4),
        })
    return sorted(
        [r for r in results if r["Score antigenicidad"] is not None],
        key=lambda x: x["Score antigenicidad"], reverse=True
    ) + [r for r in results if r["Score antigenicidad"] is None]


def plot_scores(results):
    valid = [r for r in results if r["Score antigenicidad"] is not None]
    if not valid:
        return None
    names  = [r["Proteína"][:40] for r in valid]
    scores = [r["Score antigenicidad"] for r in valid]
    colors = ["#5cb85c" if s >= 0.5 else "#d9534f" for s in scores]
    fig, ax = plt.subplots(figsize=(9, max(3, len(names) * 0.5 + 1)))
    ax.barh(names[::-1], scores[::-1], color=colors[::-1], height=0.6)
    ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=1, label="Umbral (0.5)")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Score de antigenicidad", fontsize=11)
    ax.set_title("Predicción de antigenicidad por proteína", fontsize=13)
    ax.legend(fontsize=9)
    for i, (name, score) in enumerate(zip(names[::-1], scores[::-1])):
        ax.text(score + 0.01, i, f"{score:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    return fig


# ── Interfaz ─────────────────────────────────────────────────────────────────
st.title("🧬 Clasificador de Antigenicidad de Proteínas")
st.markdown(
    "Herramienta educativa de Machine Learning para priorizar candidatos antigénicos "
    "a partir de secuencias de proteínas en formato FASTA."
)

with st.sidebar:
    st.header("ℹ️ Información del modelo")
    st.metric("AUC-ROC (CV k=5)", f"{auc_cv:.3f}")
    st.metric("Proteínas de entrenamiento", f"{n_train:,}")
    st.markdown("---")
    st.markdown("**Modelo:** Random Forest (200 árboles)")
    st.markdown("**Features:** 24 (4 fisicoquímicas + 20 composición aa)")
    st.markdown("**Patógenos de entrenamiento:** SARS-CoV-2, Influenza A")
    st.markdown("**Fuente de datos:** IEDB (tcell + bcell)")
    st.markdown("---")
    st.markdown(
        "⚠️ **Limitaciones**\n\n"
        "- Solo secuencia de aminoácidos, sin estructura 3D\n"
        "- Entrenado en 2 patógenos\n"
        "- No es un predictor clínico\n"
        "- Score alto no garantiza buen antígeno vacunal"
    )

st.markdown("### Introduce las secuencias proteicas")
col1, col2 = st.columns([3, 1])

with col1:
    fasta_input = st.text_area(
        "Pega aquí tu archivo FASTA:",
        height=250,
        placeholder=">Nombre_proteina\nSECUENCIAAMINOACIDOS...",
        help="Formato FASTA estándar. Una o más proteínas."
    )

with col2:
    st.markdown("**O sube un archivo .fasta**")
    uploaded_file = st.file_uploader(
        "Archivo FASTA", type=["fasta", "fa", "txt"],
        label_visibility="collapsed"
    )
    if uploaded_file is not None:
        fasta_input = uploaded_file.read().decode("utf-8")
        st.success(f"Archivo cargado: {uploaded_file.name}")
    st.markdown("---")
    st.markdown("**Ejemplo rápido**")
    if st.button("Cargar proteínas SARS-CoV-2", use_container_width=True):
        fasta_input = (
            ">Spike_glycoprotein\nMFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHV\n"
            ">Nucleoprotein\nMSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQ\n"
            ">Membrane_protein\nMADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVY\n"
            ">Envelope_protein\nMYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSR\n"
        )
        st.rerun()

if st.button("🔬 Predecir antigenicidad", type="primary", use_container_width=True):
    if not fasta_input or not fasta_input.strip():
        st.error("Por favor introduce al menos una secuencia en formato FASTA.")
    else:
        proteins = parse_fasta(fasta_input)
        if not proteins:
            st.error("No se encontraron secuencias válidas. Verifica el formato FASTA.")
        else:
            with st.spinner(f"Analizando {len(proteins)} proteína(s)..."):
                results = predict_proteins(proteins)
            st.markdown("---")
            st.markdown(f"### Resultados — {len(proteins)} proteína(s) analizadas")
            fig = plot_scores(results)
            if fig:
                st.pyplot(fig)
                plt.close()
            import pandas as pd
            df_results = pd.DataFrame(results)
            st.dataframe(
                df_results, use_container_width=True, hide_index=True,
                column_config={
                    "Score antigenicidad": st.column_config.ProgressColumn(
                        "Score antigenicidad", min_value=0, max_value=1, format="%.4f"
                    )
                }
            )
            csv = df_results.to_csv(index=False).encode("utf-8")
            st.download_button(
                "⬇️ Descargar resultados (CSV)", data=csv,
                file_name="antigenicidad_resultados.csv", mime="text/csv"
            )
            with st.expander("ℹ️ Cómo interpretar los resultados"):
                st.markdown(
                    "**Score de antigenicidad:** probabilidad estimada de que la proteína "
                    "sea reconocida por el sistema inmune humano (0-1). Umbral: **0.5**.\n\n"
                    "Esta herramienta es un **filtro orientativo**, no un predictor clínico. "
                    "La validación experimental es siempre necesaria."
                )

Writing /content/app/app.py


## 4. Lanzar la app con ngrok

Streamlit necesita un túnel público para ser accesible desde el navegador
cuando se ejecuta en Colab. Usamos ngrok para esto.

**Antes de ejecutar esta celda:**
1. Crea una cuenta gratuita en [https://ngrok.com](https://ngrok.com)
2. Ve a *Your Authtoken* en el dashboard y copia tu token
3. Pégalo en la variable `NGROK_TOKEN` de la celda siguiente

In [5]:
import subprocess
import threading
import time
from pyngrok import ngrok, conf

# ── CONFIGURA TU TOKEN AQUÍ ──────────────────────────────────────────────────
NGROK_TOKEN = "TU_TOKEN_AQUI"   # <-- pega aquí tu authtoken de ngrok
# ─────────────────────────────────────────────────────────────────────────────

if NGROK_TOKEN == "TU_TOKEN_AQUI":
    print("⚠️  Debes pegar tu authtoken de ngrok en la variable NGROK_TOKEN")
    print("   Regístrate gratis en https://ngrok.com y copia tu token")
else:
    # Configurar ngrok
    conf.get_default().auth_token = NGROK_TOKEN

    # Lanzar Streamlit en background
    def run_streamlit():
        subprocess.run(
            ["streamlit", "run", "/content/app/app.py",
             "--server.port=8501",
             "--server.headless=true",
             "--server.enableCORS=false"],
            cwd="/content/app"
        )

    thread = threading.Thread(target=run_streamlit, daemon=True)
    thread.start()

    # Esperar a que arranque
    time.sleep(5)

    # Abrir túnel ngrok
    public_url = ngrok.connect(8501)
    print(f"\n{'='*55}")
    print(f"  App disponible en: {public_url}")
    print(f"{'='*55}")
    print("  Abre esa URL en tu navegador para usar la app.")
    print("  Mantén esta celda en ejecución mientras uses la app.")

⚠️  Debes pegar tu authtoken de ngrok en la variable NGROK_TOKEN
   Regístrate gratis en https://ngrok.com y copia tu token


## 5. Verificación de la app

Una vez abierta la app en el navegador, prueba el flujo completo:

1. Haz clic en **"Cargar proteínas SARS-CoV-2"** para cargar el ejemplo precargado
2. Haz clic en **"Predecir antigenicidad"**
3. Verifica que **Spike glycoprotein** aparece con el score más alto
4. Descarga el CSV de resultados
5. Prueba subir un archivo `.fasta` propio

Si Spike aparece arriba, la demo funciona correctamente y el modelo
captura algo biológicamente coherente.

## 6. Ejecución local (fuera de Colab)

Para ejecutar la app en una máquina local:

```bash
pip install streamlit biopython joblib scikit-learn
cd app/
streamlit run app.py
```

El archivo `model.pkl` debe estar en el mismo directorio que `app.py`.